# JetRacer Lane Fusion V2 - Live

V2 segmentation + waypoint fusion, with the original direct CSI camera and motor callback. Motor is DISARMED by default.

In [1]:
import sys, time, csv, threading
from pathlib import Path
import cv2, numpy as np, ipywidgets as widgets
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'V2' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg
from notebook3.basic_motion import JetRacerController
from V2.config import load_config
from V2.segmentation import Segmenter
from V2.geometry import estimate_geometry
from V2.fusion import fuse
from V2.control import Controller, State

cfg = load_config(PROJECT_ROOT / 'V2/config.yaml')
root = Path(cfg['_root'])
segmenter = Segmenter(cfg)
controller = Controller(cfg['control'])
model = None
model_mode = 'geometry-fallback'
try:
    from waypoint_lane_fusion.lane_model import TensorRTWaypointModel, OnnxWaypointModel
    engine = root / cfg['models'].get('waypoint_engine', 'models/waypoint_baseline_fp16.engine')
    onnx = root / cfg['models']['waypoint']
    if engine.exists(): model = TensorRTWaypointModel(engine); model_mode = 'waypoint-tensorrt-fp16'
    else: model = OnnxWaypointModel(onnx); model_mode = 'waypoint-onnx'
except Exception as exc:
    model_error = str(exc)
print('V2 perception:', model_mode)
if 'model_error' in globals(): print('Waypoint fallback:', model_error)

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


V2 perception: waypoint-tensorrt-fp16


Nếu camera đang bị khóa, chạy `sudo systemctl restart nvargus-daemon` trong terminal trước cell tiếp theo.

In [2]:
try:
    camera.running = False; camera.unobserve_all()
except Exception:
    pass
c = cfg['camera']
camera = CSICamera(width=c['width'], height=c['height'], capture_fps=0)
h = cfg['hardware']
car = JetRacerController(h['steering_gain'], h['steering_offset'], h['throttle_gain'], cfg['control']['throttle_max'])
car.stop(); car.center_steering()
print('Camera and motor controller initialized. Motor is stopped.')

[JetRacer] Đã dừng xe!
[JetRacer] Đã dừng xe!
Camera and motor controller initialized. Motor is stopped.


In [3]:
state = widgets.ToggleButtons(options=['stop', 'live'], value='stop', description='State')
armed = widgets.Checkbox(value=False, description='ARM MOTOR')
max_throttle = widgets.FloatSlider(value=cfg['control']['throttle_max'], min=0.0, max=1.0, step=0.005, description='Max throttle', continuous_update=False)
max_steering = widgets.FloatSlider(value=cfg['control']['max_steering'], min=0.0, max=1.0, step=0.01, description='Max steering', continuous_update=False)
raw_view = widgets.Image(format='jpeg', width=224, height=224)
debug_view = widgets.Image(format='jpeg', width=224, height=224)
status = widgets.HTML(value='<b>STOPPED / DISARMED</b>')
blank = np.zeros((224, 224, 3), np.uint8); blank_jpeg = bgr8_to_jpeg(blank)
raw_view.value = blank_jpeg; debug_view.value = blank_jpeg
display(widgets.VBox([widgets.HBox([raw_view, debug_view]), status, widgets.HBox([state, armed]), max_throttle, max_steering]))

In [4]:
callback_lock = threading.Lock(); last_tick = time.perf_counter(); fps_ema = 0.0
log_dir = PROJECT_ROOT / 'V2/logs'; log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / time.strftime('live_%Y%m%d_%H%M%S.csv')
log_stream = log_path.open('w', newline=''); log_writer = csv.writer(log_stream); log_buffer = []
log_writer.writerow(['timestamp','fps','waypoint_x','waypoint_y','road_confidence','steering','throttle','state','armed'])

def set_limits(change=None):
    cfg['control']['throttle_max'] = float(max_throttle.value)
    cfg['control']['max_steering'] = float(max_steering.value)
    car.max_throttle = float(max_throttle.value)
max_throttle.observe(set_limits, names='value'); max_steering.observe(set_limits, names='value')

def v2_command(frame):
    small = cv2.resize(frame, (int(c['width']), int(c['height'])))
    (road, outside, marking), perception_mode = segmenter.infer(small)
    geom = estimate_geometry(road, outside, marking, cfg['geometry'], segmenter.obstacle)
    if model is not None:
        point = model.predict(small); waypoint = (point.x, point.y, point.confidence)
    else:
        ys, xs = np.nonzero(marking > 0); keep = ys > int(marking.shape[0] * .48)
        if np.count_nonzero(keep) >= 8:
            coef = np.polyfit(ys[keep], xs[keep], 1)
            x = np.polyval(coef, int(marking.shape[0] * .64)) / marking.shape[1]
            waypoint = (float(np.clip(x, 0, 1)), .64, min(1., np.count_nonzero(keep) / 80.))
        else: waypoint = (geom.center_x / small.shape[1], .64, 0.)
    target = fuse(waypoint, geom, small.shape[1], small.shape[0])
    now = time.perf_counter(); dt = max(.005, now - last_tick)
    command = controller.update(target, geom, dt)
    command.steering = float(np.clip(command.steering, -max_steering.value, max_steering.value))
    command.throttle = float(np.clip(command.throttle, -max_throttle.value, max_throttle.value))
    return small, road, outside, marking, geom, target, command, dt, perception_mode

def live_update(change):
    global last_tick, fps_ema
    if state.value != 'live' or not callback_lock.acquire(False): return
    try:
        frame = change['new']; result = v2_command(frame)
        small, road, outside, marking, geom, target, command, dt, perception_mode = result
        if armed.value and command.state not in (State.STOP.value, State.REACQUIRE.value):
            car.set_steering(command.steering); car.set_throttle(command.throttle)
        else:
            car.stop(); car.center_steering()
        last_tick = time.perf_counter(); instant = 1. / dt; fps_ema = instant if not fps_ema else .2 * instant + .8 * fps_ema
        rendered = small.copy(); rendered[road > 0] = (30, 110, 30); rendered[outside > 0] = (220, 220, 220); rendered[marking > 0] = (0, 80, 220); rendered[segmenter.obstacle > 0] = (0, 0, 255)
        if geom.points: cv2.polylines(rendered, [np.asarray([(int(x), int(y)) for y, x in geom.points], np.int32)], False, (0, 255, 0), 2)
        cv2.circle(rendered, (int(target.x * 224), int(target.y * 224)), 5, (255, 0, 255), -1)
        cv2.putText(rendered, '%s %.1ffps road %.2f steer %.2f gas %.2f' % (command.state, fps_ema, geom.confidence, command.steering, command.throttle), (2, 15), cv2.FONT_HERSHEY_SIMPLEX, .35, (0, 255, 255), 1)
        raw_view.value = bgr8_to_jpeg(small); debug_view.value = bgr8_to_jpeg(rendered)
        status.value = '<b>%s | %s | FPS %.1f | road %.2f | steer %+.3f | throttle %+.3f | armed=%s</b>' % (perception_mode, command.state, fps_ema, geom.confidence, command.steering, command.throttle, armed.value)
        log_buffer.append([time.time(), fps_ema, target.x, target.y, geom.confidence, command.steering, command.throttle, command.state, int(armed.value)])
        if len(log_buffer) >= 20: log_writer.writerows(log_buffer); log_stream.flush(); log_buffer.clear()
    except Exception as exc:
        car.stop(); car.center_steering(); state.value = 'stop'; status.value = '<b style="color:red">ERROR: %s</b>' % exc
    finally: callback_lock.release()

def safety_changed(change):
    if change.get('new') == 'stop' or not armed.value:
        car.stop(); car.center_steering()
state.observe(safety_changed, names='value'); armed.observe(safety_changed, names='value')
camera.observe(live_update, names='value'); camera.running = True
print('Camera running. Chon live, then ARM MOTOR. Controller is automatic within the two caps.')

Camera running. Chon live, then ARM MOTOR. Controller is automatic within the two caps.


## Dung an toan - luon chay cell tiep theo truoc khi dong notebook

In [ ]:
state.value = 'stop'; armed.value = False; car.stop(); car.center_steering()
camera.running = False; camera.unobserve_all()
if log_buffer: log_writer.writerows(log_buffer); log_buffer.clear()
log_stream.flush(); log_stream.close()
print('Stopped safely. Log:', log_path)